In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import random, numpy, pandas
from  lightgbm import LGBMRegressor, LGBMClassifier, log_evaluation, early_stopping
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

def seed_everything(seed):
    np.random.seed(seed)
    random.seed(seed)
    
seed_everything(seed=2025)

In [2]:
from catboost import CatBoostRegressor ,Pool
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.linear_model import ElasticNetCV, LassoCV, RidgeCV
from sklearn.svm import SVR

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.ensemble import AdaBoostRegressor

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import accuracy_score

from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_val_score

from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif
from sklearn.feature_selection import SelectFromModel

from sklearn.impute import SimpleImputer
import sklearn.linear_model as linear_model
from sklearn.model_selection import train_test_split
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

In [3]:

def get_next_day(date_str):
    if date_str=='None':
        return 'None'
    date_obj = datetime.strptime(date_str, '%Y-%m-%d')
    previous_day = date_obj + timedelta(days=1)
    previous_day_str = previous_day.strftime('%Y-%m-%d')
    return previous_day_str

def get_feats(mode='train'):
    path=f'/kaggle/input/phems-hackathon-early-sepsis-prediction/{mode}ing_data/'
    
    print("< feats >")
    feats=pd.read_csv(path+f"SepsisLabel_{mode}.csv").drop_duplicates()
    #merge with other tables
    feats['measurement_datetime_day']=feats['measurement_datetime'].fillna('None').apply(lambda x:x[:10])

    print("< drug >")
    drug=pd.read_csv(path+f"drugsexposure_{mode}.csv")
    drug['measurement_datetime_day']=drug['drug_datetime_hourly'].fillna('None').apply(lambda x:x[:10])
    for col in ['drug_concept_id','route_concept_id']:
        drug[col]=drug[col].fillna('None').astype(str)
        group_df=drug.groupby(['person_id','measurement_datetime_day'])[col].apply(lambda x:" ".join(x)).reset_index()
        feats=feats.merge(group_df,on=['person_id','measurement_datetime_day'],how='left')
        feats[col]=feats[col].fillna('None')
    
    drug['measurement_datetime_day']=drug['measurement_datetime_day'].apply(lambda x:get_next_day(x))
    for col in ['drug_concept_id','route_concept_id']:
        group_df=drug.groupby(['person_id','measurement_datetime_day'])[col].apply(lambda x:" ".join(x)).reset_index().rename(columns={col:col+"previous_day"})
        feats=feats.merge(group_df,on=['person_id','measurement_datetime_day'],how='left')
        feats[col+"previous_day"]=feats[col+"previous_day"].fillna('None')
    
    feats['measurement_datetime']=pd.to_datetime(feats['measurement_datetime'])
    feats['dow']=feats['measurement_datetime'].dt.dayofweek
    feats['doy']=feats['measurement_datetime'].dt.dayofyear
    feats['hour']=feats['measurement_datetime'].dt.hour
    feats.drop(['measurement_datetime'],axis=1,inplace=True)

    print("-"*30)
    return feats

train=get_feats(mode='train')
test=get_feats(mode='test')
print(train.shape,test.shape)

< feats >
< drug >
------------------------------
< feats >
< drug >
------------------------------
(331639, 10) (130483, 9)


In [4]:
encoder = LabelEncoder()
#normalize = SimpleImputer(strategy='mean')

for i in zip(train.columns,train.dtypes):
    if i[1]=='O':
        train[i[0]] = train[i[0]].fillna('Unknown')
        train[i[0]]=encoder.fit_transform(train[i[0]].to_numpy().reshape(-1,1))
    else:
        train[i[0]] = train[i[0]].fillna(0)

for i in zip(test.columns,test.dtypes):
    if i[1]=='O':
        test[i[0]] = test[i[0]].fillna('Unknown')
        test[i[0]]=encoder.fit_transform(test[i[0]].to_numpy().reshape(-1,1))
    else:
        test[i[0]] = test[i[0]].fillna(0)

In [5]:
train_x = train.drop(columns=['SepsisLabel', 'person_id'])
train_y = train['SepsisLabel']

In [6]:
train.head()

,person_id,SepsisLabel,measurement_datetime_day,drug_concept_id,route_concept_id,drug_concept_idprevious_day,route_concept_idprevious_day,dow,doy,hour
0,274096387,0,2144,1521,81,1440,95,1.0,338.0,20.0
1,1719359031,0,1917,2760,1504,2638,1466,5.0,111.0,9.0
2,2024544816,0,906,2143,126,0,1000,2.0,195.0,7.0
3,213710896,0,1220,0,1019,7614,537,1.0,144.0,7.0
4,1335786468,0,2044,0,1019,0,1000,6.0,238.0,22.0


In [7]:
Fold=5
kf = KFold(n_splits=Fold)

cat_pre_test = numpy.zeros(len(test))

catboost = CatBoostRegressor(grow_policy='Depthwise',
                             iterations=1000,
                             eval_metric='RMSE',
                             random_state=0,
                             verbose=100)

for i, (train_index, test_index) in enumerate(kf.split(train)):

    print(f"FOLD: {i}")

    x_fold=train_x[train_index[0]: train_index[len(train_index)-1]]
    y_fold=train_y[train_index[0]: train_index[len(train_index)-1]]

    x_test_fold=train_x[test_index[0]: test_index[len(test_index)-1]]
    y_test_fold=train_y[test_index[0]: test_index[len(test_index)-1]]

    cat_features = list(train_x.select_dtypes(include=['object', 'category']).columns)
    
    train_pool = Pool(x_fold, y_fold, cat_features=cat_features)
    val_pool = Pool(x_test_fold, y_test_fold, cat_features=cat_features)
    
    #weight_ = weight[train_index[0]: train_index[len(train_index)-1]]
    
    catboost.fit(train_pool, eval_set=(val_pool), early_stopping_rounds=100)
    
cat_pre_test=catboost.predict(test)
print(catboost.score(x_test_fold, y_test_fold))

FOLD: 0
Learning rate set to 0.1223
0:	learn: 0.1361853	test: 0.1373926	best: 0.1373926 (0)	total: 114ms	remaining: 1m 54s
100:	learn: 0.0667069	test: 0.0705525	best: 0.0705525 (100)	total: 4.43s	remaining: 39.5s
200:	learn: 0.0526870	test: 0.0583728	best: 0.0583728 (200)	total: 8.67s	remaining: 34.5s
300:	learn: 0.0453600	test: 0.0520000	best: 0.0520000 (300)	total: 12.8s	remaining: 29.8s
400:	learn: 0.0404231	test: 0.0481593	best: 0.0481593 (400)	total: 17.1s	remaining: 25.5s
500:	learn: 0.0363240	test: 0.0449482	best: 0.0449482 (500)	total: 21.2s	remaining: 21.1s
600:	learn: 0.0335566	test: 0.0428260	best: 0.0428260 (600)	total: 25.4s	remaining: 16.9s
700:	learn: 0.0303745	test: 0.0402810	best: 0.0402810 (700)	total: 30.2s	remaining: 12.9s
800:	learn: 0.0281637	test: 0.0386848	best: 0.0386848 (800)	total: 34.6s	remaining: 8.58s
900:	learn: 0.0260956	test: 0.0370772	best: 0.0370772 (900)	total: 38.8s	remaining: 4.27s
999:	learn: 0.0248061	test: 0.0362199	best: 0.0362199 (999)	total: 

In [8]:
id = pandas.read_csv('/kaggle/input/phems-hackathon-early-sepsis-prediction/SepsisLabel_sample_submission.csv')['person_id_datetime']
test_predictions = (cat_pre_test)

In [9]:
test_predictions

array([-0.01079801, -0.04021932, -0.01394999, ..., -0.0171772 ,
       -0.00836685, -0.02226123])

In [10]:
submission = pandas.DataFrame({
    'ID': id.values,
    'prediction': test_predictions,
})
submission

,ID,prediction
0,1416048048_2021-03-25 10:00:00,-0.010798
1,280531880_2024-01-22 18:00:00,-0.040219
2,1127023302_2023-12-29 21:00:00,-0.013950
3,2065909112_2021-07-07 05:00:00,0.117998
4,264445818_2024-08-23 22:00:00,-0.052175
...,...,...
130478,1968006557_2023-10-28 03:00:00,0.046132
130479,1511876642_2024-09-04 15:00:00,0.057043
130480,1844025915_2024-05-09 10:00:00,-0.017177
130481,264445818_2024-07-07 15:00:00,-0.008367


In [11]:
submission.to_csv('submission.csv', index = False)